# 04 — NLP News Processing

**Project:** AI Supply Chain Digital Marketing

This notebook processes the supplied `cleaned_news.csv` dataset.

Pipeline:
1. Load and verify cleaned news data
2. Clean news text
3. Remove stopwords
4. Lemmatize text
5. Encode existing sentiment labels
6. Extract supply-chain event signals
7. Generate TF-IDF features
8. Save processed NLP datasets

**Important:** The supplied news dataset has no date column, so no artificial news dates are created.

In [1]:
import re
import string
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Project paths
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NEWS_FILE = INTERIM_DIR / "cleaned_news.csv"

print("Project directory:", PROJECT_DIR)
print("News file:", NEWS_FILE)

Project directory: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
News file: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\interim\cleaned_news.csv


In [3]:
# Load cleaned news dataset
news = pd.read_csv(
    NEWS_FILE,
    encoding="latin1"
)

print("News shape:", news.shape)
print("Columns:", news.columns.tolist())

News shape: (4840, 2)
Columns: ['sentiment', 'text']


In [4]:
# Verify required columns
required_columns = ["sentiment", "text"]

missing_columns = [
    col for col in required_columns
    if col not in news.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Required columns verified.")

Required columns verified.


In [5]:
# Inspect sample records
display(news.head(10))

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...
5,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...
6,positive,"For the last quarter of 2010 , Componenta 's n..."
7,positive,"In the third quarter of 2010 , net sales incre..."
8,positive,Operating profit rose to EUR 13.1 mn from EUR ...
9,positive,"Operating profit totalled EUR 21.1 mn , up fro..."


In [6]:
# Missing-value check
missing_values = news[["sentiment", "text"]].isna().sum()

print("Missing values:")
display(missing_values)

Missing values:


sentiment    0
text         0
dtype: int64

In [7]:
# Sentiment labels
print("Unique sentiment labels:")
print(news["sentiment"].unique())

print("\nSentiment distribution:")
display(news["sentiment"].value_counts())

Unique sentiment labels:
['neutral' 'negative' 'positive']

Sentiment distribution:


sentiment
neutral     2873
positive    1363
negative     604
Name: count, dtype: int64

In [8]:
# Convert text to string and remove empty text records
news["text"] = news["text"].astype(str).str.strip()
news = news[news["text"] != ""].copy()

print("Shape after removing empty text:", news.shape)

Shape after removing empty text: (4840, 2)


## Text Cleaning

In [9]:
def clean_text(text):
    text = str(text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


news["clean_text"] = news["text"].apply(clean_text)

print("Text cleaning completed.")

Text cleaning completed.


In [10]:
# Compare original and cleaned text
display(news[["text", "clean_text"]].head(10))

,text,clean_text
0,"According to Gran , the company has no plans t...",according to gran the company has no plans to ...
1,Technopolis plans to develop in stages an area...,technopolis plans to develop in stages an area...
2,The international electronic industry company ...,the international electronic industry company ...
3,With the new production plant the company woul...,with the new production plant the company woul...
4,According to the company 's updated strategy f...,according to the company s updated strategy fo...
5,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,financing of aspocomp s growth aspocomp is agg...
6,"For the last quarter of 2010 , Componenta 's n...",for the last quarter of componenta s net sales...
7,"In the third quarter of 2010 , net sales incre...",in the third quarter of net sales increased by...
8,Operating profit rose to EUR 13.1 mn from EUR ...,operating profit rose to eur mn from eur mn in...
9,"Operating profit totalled EUR 21.1 mn , up fro...",operating profit totalled eur mn up from eur m...


In [11]:
# Basic text statistics
news["text_length"] = news["clean_text"].str.len()
news["word_count"] = news["clean_text"].str.split().str.len()

display(news[["text_length", "word_count"]].describe())

,text_length,word_count
count,4840.000000,4840.000000
mean,116.369421,19.369421
std,52.698767,8.356436
min,7.000000,1.000000
25%,75.000000,13.000000
50%,108.000000,18.000000
75%,151.000000,25.000000
max,297.000000,50.000000


## Stopword Removal

In [13]:
# Install NLTK if it is not already available in the environment
try:
    import nltk
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK datasets/resources
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("NLTK imported.")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("NLTK imported.")

NLTK imported.
NLTK imported.


In [14]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources ready.")

NLTK resources ready.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [15]:
stop_words = set(stopwords.words("english"))

print("Number of stopwords:", len(stop_words))

Number of stopwords: 198


In [16]:
def remove_stopwords(text):
    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)


news["no_stopwords"] = news["clean_text"].apply(remove_stopwords)

print("Stopword removal completed.")

Stopword removal completed.


## Lemmatization

In [17]:
lemmatizer = WordNetLemmatizer()

print("Lemmatizer initialized.")

Lemmatizer initialized.


In [18]:
def lemmatize_text(text):
    words = text.split()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(lemmatized_words)


news["processed_text"] = news["no_stopwords"].apply(
    lemmatize_text
)

print("Lemmatization completed.")

Lemmatization completed.


In [19]:
# Compare preprocessing stages
display(
    news[
        [
            "text",
            "clean_text",
            "no_stopwords",
            "processed_text"
        ]
    ].head(10)
)

,text,clean_text,no_stopwords,processed_text
0,"According to Gran , the company has no plans t...",according to gran the company has no plans to ...,according gran company plans move production r...,according gran company plan move production ru...
1,Technopolis plans to develop in stages an area...,technopolis plans to develop in stages an area...,technopolis plans develop stages area less squ...,technopolis plan develop stage area less squar...
2,The international electronic industry company ...,the international electronic industry company ...,international electronic industry company elco...,international electronic industry company elco...
3,With the new production plant the company woul...,with the new production plant the company woul...,new production plant company would increase ca...,new production plant company would increase ca...
4,According to the company 's updated strategy f...,according to the company s updated strategy fo...,according company updated strategy years baswa...,according company updated strategy year baswar...
5,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,financing of aspocomp s growth aspocomp is agg...,financing aspocomp growth aspocomp aggressivel...,financing aspocomp growth aspocomp aggressivel...
6,"For the last quarter of 2010 , Componenta 's n...",for the last quarter of componenta s net sales...,last quarter componenta net sales doubled eur ...,last quarter componenta net sale doubled eur e...
7,"In the third quarter of 2010 , net sales incre...",in the third quarter of net sales increased by...,third quarter net sales increased eur mn opera...,third quarter net sale increased eur mn operat...
8,Operating profit rose to EUR 13.1 mn from EUR ...,operating profit rose to eur mn from eur mn in...,operating profit rose eur mn eur mn correspond...,operating profit rose eur mn eur mn correspond...
9,"Operating profit totalled EUR 21.1 mn , up fro...",operating profit totalled eur mn up from eur m...,operating profit totalled eur mn eur mn repres...,operating profit totalled eur mn eur mn repres...


## NLP Feature Extraction

In [20]:
# Processed text statistics
news["processed_word_count"] = (
    news["processed_text"]
    .str.split()
    .str.len()
)

news["processed_char_count"] = (
    news["processed_text"]
    .str.len()
)

print("Basic NLP features created.")

Basic NLP features created.


In [21]:
# Encode existing sentiment labels
sentiment_mapping = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

news["sentiment_score"] = (
    news["sentiment"]
    .str.lower()
    .map(sentiment_mapping)
)

display(
    news[["sentiment", "sentiment_score"]].head(10)
)

,sentiment,sentiment_score
0,neutral,0
1,neutral,0
2,negative,-1
3,positive,1
4,positive,1
5,positive,1
6,positive,1
7,positive,1
8,positive,1
9,positive,1


In [22]:
# Validate sentiment encoding
invalid_sentiment = news["sentiment_score"].isna().sum()

print("Invalid sentiment records:", invalid_sentiment)

if invalid_sentiment > 0:
    raise ValueError("Unexpected sentiment labels found.")

print("All sentiment labels successfully encoded.")

Invalid sentiment records: 0
All sentiment labels successfully encoded.


## Supply-Chain Event Signal Extraction

In [23]:
event_keywords = {
    "war": [
        "war", "conflict", "invasion", "military",
        "attack", "missile", "battle"
    ],

    "strike": [
        "strike", "strikes", "worker strike",
        "labor strike", "walkout"
    ],

    "weather": [
        "storm", "hurricane", "flood", "cyclone",
        "typhoon", "earthquake", "weather"
    ],

    "tariff": [
        "tariff", "tariffs", "trade war",
        "import duty", "export duty"
    ],

    "port": [
        "port", "port closure", "port congestion",
        "container", "terminal"
    ],

    "shipping": [
        "shipping", "freight", "vessel",
        "cargo", "shipment", "shipping rate"
    ],

    "pandemic": [
        "pandemic", "covid", "coronavirus",
        "lockdown"
    ]
}

print("Event keyword groups:", list(event_keywords.keys()))

Event keyword groups: ['war', 'strike', 'weather', 'tariff', 'port', 'shipping', 'pandemic']


In [24]:
def detect_events(text):
    text = str(text).lower()

    detected = []

    for event, keywords in event_keywords.items():
        if any(keyword in text for keyword in keywords):
            detected.append(event)

    return detected


news["detected_events"] = news["processed_text"].apply(
    detect_events
)

display(
    news[["text", "detected_events"]].head(10)
)

,text,detected_events
0,"According to Gran , the company has no plans t...",[]
1,Technopolis plans to develop in stages an area...,[]
2,The international electronic industry company ...,[port]
3,With the new production plant the company woul...,[]
4,According to the company 's updated strategy f...,[war]
5,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,[]
6,"For the last quarter of 2010 , Componenta 's n...",[]
7,"In the third quarter of 2010 , net sales incre...",[]
8,Operating profit rose to EUR 13.1 mn from EUR ...,[]
9,"Operating profit totalled EUR 21.1 mn , up fro...",[]


In [25]:
# Create binary event features
for event in event_keywords.keys():
    news[f"event_{event}"] = news["detected_events"].apply(
        lambda x: int(event in x)
    )

event_columns = [
    f"event_{event}"
    for event in event_keywords.keys()
]

print("Event indicator features created.")

Event indicator features created.


In [26]:
# Event summary
event_summary = (
    news[event_columns]
    .sum()
    .sort_values(ascending=False)
    .to_frame("news_count")
)

display(event_summary)

,news_count
event_port,415
event_war,229
event_shipping,81
event_weather,9
event_strike,7
event_tariff,2
event_pandemic,0


## TF-IDF Feature Extraction

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    news["processed_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4840, 1000)


In [28]:
# Convert TF-IDF matrix to dataframe
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[
        f"tfidf_{feature}"
        for feature in tfidf_feature_names
    ],
    index=news.index
)

print("TF-IDF dataframe shape:", tfidf_df.shape)

TF-IDF dataframe shape: (4840, 1000)


## Create Final NLP Feature Dataset

In [29]:
nlp_features = news[
    [
        "sentiment_score",
        "text_length",
        "word_count",
        "processed_word_count",
        "processed_char_count"
    ] + event_columns
].copy()

print("NLP statistical/event features:")
print(nlp_features.shape)

display(nlp_features.head())

NLP statistical/event features:
(4840, 12)


,sentiment_score,text_length,word_count,processed_word_count,processed_char_count,event_war,event_strike,event_weather,event_tariff,event_port,event_shipping,event_pandemic
0,0,121,22,10,75,0,0,0,0,0,0,0
1,0,178,28,17,133,0,0,0,0,0,0,0
2,-1,222,33,21,171,0,0,0,0,1,0,0
3,1,204,32,20,157,0,0,0,0,0,0,0
4,1,165,30,17,118,1,0,0,0,0,0,0


In [30]:
# Combine statistical/event features with TF-IDF
news_nlp_features = pd.concat(
    [
        nlp_features,
        tfidf_df
    ],
    axis=1
)

print("Final NLP feature matrix shape:")
print(news_nlp_features.shape)

Final NLP feature matrix shape:
(4840, 1012)


In [31]:
# Save NLP feature matrix
output_file = PROCESSED_DIR / "news_nlp_features.csv"

news_nlp_features.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\news_nlp_features.csv


In [32]:
# Save processed news text and interpretable NLP features
processed_news_file = PROCESSED_DIR / "processed_news.csv"

news[
    [
        "sentiment",
        "text",
        "clean_text",
        "no_stopwords",
        "processed_text",
        "sentiment_score",
        "text_length",
        "word_count",
        "detected_events"
    ] + event_columns
].to_csv(
    processed_news_file,
    index=False
)

print("Saved:", processed_news_file)

Saved: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\processed\processed_news.csv


## Final Validation

In [33]:
missing_nlp = news_nlp_features.isna().sum().sum()

print("Total missing cells in NLP features:", missing_nlp)

Total missing cells in NLP features: 0


In [34]:
numeric_nlp = news_nlp_features.select_dtypes(
    include=np.number
)

infinite_values = np.isinf(
    numeric_nlp.to_numpy()
).sum()

print("Infinite values:", infinite_values)

Infinite values: 0


In [35]:
print("=" * 60)
print("FINAL NLP PROCESSING SUMMARY")
print("=" * 60)

print("Original/processed news records:", len(news))
print("NLP feature rows:", len(news_nlp_features))
print("NLP feature columns:", news_nlp_features.shape[1])
print("Missing NLP values:", missing_nlp)
print("Infinite NLP values:", infinite_values)
print("TF-IDF features:", len(tfidf_feature_names))
print("Event feature count:", len(event_columns))

print("=" * 60)
print("NLP PROCESSING COMPLETED")
print("=" * 60)

FINAL NLP PROCESSING SUMMARY
Original/processed news records: 4840
NLP feature rows: 4840
NLP feature columns: 1012
Missing NLP values: 0
Infinite NLP values: 0
TF-IDF features: 1000
Event feature count: 7
NLP PROCESSING COMPLETED


In [36]:
# Final generated-file check
print("Generated files:")

for file in [
    PROCESSED_DIR / "news_nlp_features.csv",
    PROCESSED_DIR / "processed_news.csv"
]:
    print(file.name, "->", file.exists())

Generated files:
news_nlp_features.csv -> True
processed_news.csv -> True
